In [1]:
# ============================================================================
# 04_extractor_validation.ipynb
# Specify the LLM claim extractor and establish that it can be trusted as a
# measurement instrument before the main experiment. Three things are checked:
#   (1) self-stability  - does the extractor return the same claims on repeated
#                         runs of the same explanation?
#   (2) fidelity to the supplied evidence on the pilot corpus;
#   (3) the pre-registered human-coding validation gates (inter-coder kappa and
#       extractor-vs-gold agreement), carried from the methodology pilot.
# No LLM calls are made here; the pilot extractions, repeat runs, and the
# adjudicated human-coding summary are read from data/reference.
# ============================================================================

# [Imports and global configuration]
import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ROOT = Path("..")
DATA_DIR = ROOT / "data"
REF_DIR = DATA_DIR / "reference"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3", "image.cmap": "gray",
    "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

# [Extractor configuration]
# The extractor model is read from the environment so the published code is
# key-free; the strong tier is used deterministically for extraction.
with open(REF_DIR / "extractor_config.json", encoding="utf-8") as f:
    extractor_config = json.load(f)
extractor_config["extractor_model"] = os.getenv("LLM_MODEL_STRONG", "gpt-5.4")
extractor_config["extractor_temperature"] = None
GATE_FEATURE_KAPPA, GATE_DIRECTION_KAPPA, GATE_RANK_KAPPA = 0.80, 0.80, 0.70

# [1) Self-stability from repeated extractor runs]
with open(REF_DIR / "extractor_stability.json", encoding="utf-8") as f:
    stability = json.load(f)

def feat_seq(claims):
    return [c["feature"] for c in claims]
def dir_map(claims):
    return {c["feature"]: c["direction"] for c in claims}

identical, jaccards, dir_agreements = [], [], []
for rec in stability:
    runs = rec["runs"]
    seqs = [tuple((c["feature"], c["direction"], c["rank"]) for c in r) for r in runs]
    identical.append(all(s == seqs[0] for s in seqs))
    pj, pd_ = [], []
    for a, b in combinations(runs, 2):
        fa, fb = set(feat_seq(a)), set(feat_seq(b))
        pj.append(len(fa & fb) / len(fa | fb) if (fa | fb) else 1.0)
        da, db = dir_map(a), dir_map(b)
        shared = set(da) & set(db)
        if shared:
            pd_.append(np.mean([da[f] == db[f] for f in shared]))
    jaccards.append(np.mean(pj) if pj else 1.0)
    dir_agreements.append(np.mean(pd_) if pd_ else 1.0)

self_stability = {
    "n_narratives": len(stability), "runs_each": len(stability[0]["runs"]),
    "identical_across_runs": round(float(np.mean(identical)), 3),
    "feature_jaccard": round(float(np.mean(jaccards)), 3),
    "direction_agreement": round(float(np.mean(dir_agreements)), 3),
}
print("self-stability:", self_stability)

# [2) Fidelity of the extractor to the supplied evidence on the pilot]
with open(REF_DIR / "pilot_extractions.json", encoding="utf-8") as f:
    pilot = json.load(f)

rows = []
for rec in pilot:
    ev_feats = {d["feature"] for d in rec["evidence"]}
    ev_dir = {d["feature"]: d["direction"] for d in rec["evidence"]}
    cl_feats = {c["feature"] for c in rec["claims"] if c["feature"] != "OUT_OF_LIST"}
    cl_dir = {c["feature"]: c["direction"] for c in rec["claims"]}
    tp = len(ev_feats & cl_feats)
    recall = tp / len(ev_feats) if ev_feats else np.nan
    precision = tp / len(cl_feats) if cl_feats else np.nan
    shared = ev_feats & cl_feats
    dir_acc = np.mean([ev_dir[f] == cl_dir.get(f) for f in shared]) if shared else np.nan
    hallucinated = len([c for c in rec["claims"] if c["feature"] == "OUT_OF_LIST"])
    rows.append({"domain": rec["domain"], "prompt": rec["prompt"],
                 "recall": recall, "precision": precision, "direction_accuracy": dir_acc,
                 "hallucinated": hallucinated})
fid = pd.DataFrame(rows)
pilot_fidelity = {
    "feature_recall": round(float(fid["recall"].mean()), 4),
    "feature_precision": round(float(fid["precision"].mean()), 4),
    "feature_f1": round(float(2 * fid["recall"].mean() * fid["precision"].mean()
                               / (fid["recall"].mean() + fid["precision"].mean())), 4),
    "direction_accuracy": round(float(fid["direction_accuracy"].mean()), 4),
    "hallucination_rate": round(float((fid["hallucinated"] > 0).mean()), 4),
}
print("pilot fidelity vs supplied evidence:", pilot_fidelity)
save_table(fid.groupby(["domain", "prompt"]).mean(numeric_only=True).round(4).reset_index(),
           "tab04_extractor_fidelity_by_condition")

# [3) Human-coding validation gates (carried from the methodology pilot)]
with open(REF_DIR / "validation_summary.json", encoding="utf-8") as f:
    human_val = json.load(f)
inter = {r["Level"]: r["Cohen's kappa"] for r in human_val["inter_coder"]}
extr = human_val["extractor"]
gates = [
    {"metric": "Feature kappa", "observed": extr["Feature kappa"],
     "coder_ceiling": inter["Feature"], "threshold": GATE_FEATURE_KAPPA},
    {"metric": "Direction kappa", "observed": extr["Direction kappa"],
     "coder_ceiling": inter["Direction"], "threshold": GATE_DIRECTION_KAPPA},
    {"metric": "Rank kappa", "observed": extr["Rank kappa"],
     "coder_ceiling": inter["Rank"], "threshold": GATE_RANK_KAPPA},
]
for g in gates:
    g["met"] = bool(g["observed"] >= g["threshold"])
proceed = all(g["met"] for g in gates)
gates_df = pd.DataFrame(gates)
save_table(gates_df, "tab05_validation_gates")
print("decision gates met:", proceed)

# [Write the validation summary consumed by the generation stage]
validation_summary = {
    "approach": extractor_config["approach"],
    "extractor_model": extractor_config["extractor_model"],
    "inter_coder": inter, "extractor": extr,
    "gates": gates, "proceed": proceed,
    "extractor_self_stability": self_stability,
    "pilot_fidelity_to_evidence": pilot_fidelity,
    "measurement_error": human_val.get("measurement_error", {}),
}
with open(DATA_DIR / "validation_summary.json", "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=1, ensure_ascii=False)
with open(DATA_DIR / "extractor_config.json", "w", encoding="utf-8") as f:
    json.dump(extractor_config, f, indent=1, ensure_ascii=False)
print("saved validation_summary.json and extractor_config.json")

# [Figure 11: extractor self-stability]
fig, ax = plt.subplots(figsize=(5.2, 3.4))
labels = ["Identical\nacross runs", "Feature\nJaccard", "Direction\nagreement"]
vals = [self_stability["identical_across_runs"], self_stability["feature_jaccard"],
        self_stability["direction_agreement"]]
ax.bar(range(3), vals, color=["0.6", "0.35", "0.2"], edgecolor="0.15", width=0.6)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(3)); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Rate"); ax.set_ylim(0, 1.1)
save_fig(fig, "fig11_extractor_stability")

# [Figure 12: extractor validation against the human gold standard]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
met = ["Feature precision", "Feature recall", "Feature F1", "Feature kappa",
       "Direction accuracy", "Rank kappa"]
obs = [extr["Feature precision"], extr["Feature recall"], extr["Feature F1"],
       extr["Feature kappa"], extr["Direction accuracy"], extr["Rank kappa"]]
ax.bar(range(len(met)), obs, color="0.4", edgecolor="0.15", width=0.62)
ax.axhline(GATE_FEATURE_KAPPA, color="0.2", lw=0.9, linestyle="--")
ax.text(len(met) - 0.5, GATE_FEATURE_KAPPA + 0.01, "gate 0.80", fontsize=7, ha="right")
for i, v in enumerate(obs):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(range(len(met))); ax.set_xticklabels(met, rotation=25, ha="right", fontsize=7)
ax.set_ylabel("Score"); ax.set_ylim(0.75, 1.01)
save_fig(fig, "fig12_extractor_validation")

# [Figure 13: extractor fidelity to supplied evidence by prompt]
by_prompt = fid.groupby("prompt")[["recall", "precision", "direction_accuracy"]].mean()
fig, ax = plt.subplots(figsize=(6.0, 3.4))
x = np.arange(len(by_prompt)); w = 0.26
for j, col in enumerate(["recall", "precision", "direction_accuracy"]):
    ax.bar(x + (j - 1) * w, by_prompt[col].values, width=w,
           color=["0.6", "0.4", "0.2"][j], edgecolor="0.15", label=col)
ax.set_xticks(x); ax.set_xticklabels(by_prompt.index, fontsize=8)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, frameon=False, ncol=3, loc="lower center")
save_fig(fig, "fig13_extractor_fidelity_by_prompt")

print("\nExtractor validation complete. proceed =", proceed)


self-stability: {'n_narratives': 50, 'runs_each': 3, 'identical_across_runs': 0.94, 'feature_jaccard': 1.0, 'direction_agreement': 0.99}
pilot fidelity vs supplied evidence: {'feature_recall': 0.9506, 'feature_precision': 1.0, 'feature_f1': 0.9747, 'direction_accuracy': 0.9914, 'hallucination_rate': 0.0}
saved table: tab04_extractor_fidelity_by_condition
saved table: tab05_validation_gates
decision gates met: True
saved validation_summary.json and extractor_config.json


saved figure: fig11_extractor_stability
saved figure: fig12_extractor_validation


saved figure: fig13_extractor_fidelity_by_prompt

Extractor validation complete. proceed = True
